# Part 2 — Defining the Forecasting Problem
**Appliance Energy Use Forecasting — 7PAM2033**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/DolapoMichael/Time-Series-coding-Case-study-and-Report/blob/main/notebooks/02_problem_definition.ipynb)

States the forecasting problem (target, horizon, train/test split, evaluation metrics) and defines the shared metric functions used by every later part. This notebook is self-contained — it regenerates the hourly dataset itself, so it doesn't depend on Part 1's notebook having been run first in the same session.

In [ ]:
!pip install -q statsmodels

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

RAW_CSV_URL = 'https://raw.githubusercontent.com/LuisM78/Appliances-energy-prediction-data/master/energydata_complete.csv'

DATA_DIR = Path('data')
OUTPUT_DIR = Path('outputs')
DATA_DIR.mkdir(exist_ok=True)
OUTPUT_DIR.mkdir(exist_ok=True)

## Load the hourly dataset
Reuses a local `data/energydata_hourly.csv` if Part 1 already produced one in this environment; otherwise rebuilds it from the raw source (same sum/mean logic as Part 1).

In [ ]:
hourly_path = DATA_DIR / 'energydata_hourly.csv'

if hourly_path.exists():
    hourly = pd.read_csv(hourly_path, index_col=0, parse_dates=True)
    print('Loaded existing hourly dataset from', hourly_path)
else:
    print('No local hourly dataset found — rebuilding from the raw source...')
    raw = pd.read_csv(RAW_CSV_URL)
    raw['date'] = pd.to_datetime(raw['date'])
    raw = raw.set_index('date').sort_index()

    energy_cols = ['Appliances', 'lights']
    sensor_cols = [c for c in raw.columns if c not in energy_cols + ['rv1', 'rv2']]
    hourly = pd.concat([
        raw[energy_cols].resample('h').sum(),
        raw[sensor_cols].resample('h').mean(),
    ], axis=1)
    hourly.to_csv(hourly_path)

print(f'Hourly shape: {hourly.shape}, range: {hourly.index.min()} to {hourly.index.max()}')
hourly.head()

## Problem definition
The constants below **are** the definition — change them here and every later part (benchmarks, SARIMAX, feature model, foundation model) picks up the change.

In [ ]:
TARGET = 'Appliances'          # total appliance energy use (Wh) per hour
HORIZON = 24                   # forecast horizon: 24 hours ahead
DAILY_PERIOD = 24              # observations per day at hourly resolution
WEEKLY_PERIOD = 168            # observations per week at hourly resolution
TEST_DAYS = 14                 # hold out the final 14 days as the test set
TEST_STEPS = TEST_DAYS * DAILY_PERIOD  # = 336 hourly observations

def train_test_split_by_days(series: pd.Series, test_days: int = TEST_DAYS):
    """Chronological hold-out split — not random, since shuffling would leak
    future information into training for a time-dependent problem."""
    test_steps = test_days * DAILY_PERIOD
    return series.iloc[:-test_steps], series.iloc[-test_steps:]

train, test = train_test_split_by_days(hourly[TARGET], TEST_DAYS)
print(f'Train: {train.index.min()} to {train.index.max()}  ({len(train)} obs)')
print(f'Test:  {test.index.min()} to {test.index.max()}  ({len(test)} obs)')

## Evaluation metrics: MAE, RMSE, MASE, Bias

In [ ]:
def mae(y_true, y_pred):
    """Mean Absolute Error, in the target's original units (Wh)."""
    return float(np.mean(np.abs(y_true.values - y_pred.values)))

def rmse(y_true, y_pred):
    """Root Mean Squared Error, in Wh — penalises large errors more than MAE."""
    return float(np.sqrt(np.mean((y_true.values - y_pred.values) ** 2)))

def mase(y_true, y_pred, y_train, seasonality=DAILY_PERIOD):
    """Mean Absolute Scaled Error, scaled against an in-sample seasonal-naive
    error. Below 1.0 means the model beats 'assume today looks like yesterday.'"""
    y_train = y_train.astype(float)
    naive_errors = np.abs(y_train.iloc[seasonality:].values - y_train.iloc[:-seasonality].values)
    scale = naive_errors.mean()
    if scale == 0:
        return float('nan')
    return float(np.mean(np.abs(y_true.values - y_pred.values)) / scale)

def bias(y_true, y_pred):
    """Mean signed error — positive means systematic over-forecasting."""
    return float(np.mean(y_pred.values - y_true.values))

def evaluate_forecast(name, y_true, y_pred, y_train):
    """Compute all four required metrics for one model's forecast."""
    y_pred = y_pred.reindex(y_true.index)
    valid = y_true.notna() & y_pred.notna()
    y_true_v, y_pred_v = y_true.loc[valid], y_pred.loc[valid]
    return {
        'model': name,
        'MAE': mae(y_true_v, y_pred_v),
        'RMSE': rmse(y_true_v, y_pred_v),
        'MASE': mase(y_true_v, y_pred_v, y_train, seasonality=DAILY_PERIOD),
        'Bias': bias(y_true_v, y_pred_v),
        'n_points': int(valid.sum()),
    }

In [ ]:
# Sanity check: a perfect forecast should score MAE = RMSE = MASE = Bias = 0
perfect = evaluate_forecast('perfect_forecast_sanity_check', test, test, train)
print(perfect)
assert abs(perfect['MAE']) < 1e-9
assert abs(perfect['MASE']) < 1e-9
print('\nSanity check passed.')

## Written problem statement (for the report)

In [ ]:
problem_statement = f'''# Forecasting Problem Definition

**Target variable:** `{TARGET}` — total household appliance energy use, in Wh,
aggregated to hourly resolution (summed across the six 10-minute meter readings
that fall in each hour).

**Forecast horizon:** {HORIZON} hours (one day ahead).

**Train/test split:** chronological hold-out — final {TEST_DAYS} days as test.
  - Train: {train.index.min()} to {train.index.max()}  ({len(train)} observations)
  - Test:  {test.index.min()} to {test.index.max()}  ({len(test)} observations)

**Evaluation metrics:** MAE, RMSE, MASE (scaled against 24h seasonal-naive), and Bias.

**Note on the target's construction:** the supplementary demo pipeline resamples
every column, including the target, with `.mean()`. This project instead **sums**
the `Appliances`/`lights` columns when resampling to hourly, since they are metered
Wh readings per 10-minute interval — summing six of them gives a genuine hourly Wh
total, whereas averaging returns a differently-scaled quantity. Documented here so
it is traceable in the final report.
'''

print(problem_statement)
(OUTPUT_DIR / 'problem_definition.md').write_text(problem_statement)
print('Saved to', OUTPUT_DIR / 'problem_definition.md')